# Llama-3.2-3B QLoRA Fine-Tuning on Colab

**⚠️ Runtime → Change runtime type → T4 GPU**

## 1. Install

In [ ]:
!pip install -q -U pip
!pip install -q transformers==4.45.0 accelerate==0.34.0 datasets==2.21.0 peft==0.12.0 trl==0.10.0
!pip install -q bitsandbytes>=0.43.0
print("✅ Done! RESTART: Runtime → Restart runtime")

## 2. Verify

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print("✅ Ready!")

## 3. Login & Dataset

In [ ]:
from huggingface_hub import login
login()
# Upload merged_sft_dataset.jsonl

## 4. Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Loaded")

## 5. LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files='merged_sft_dataset.jsonl', split='train').train_test_split(test_size=0.05)

def fmt(ex):
    try:
        text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
        return {'text': text}
    except:
        text = ""
        for msg in ex['messages']:
            text += f"{msg['role']}: {msg['content']}\n"
        return {'text': text}

train_ds = ds['train'].map(fmt, remove_columns=ds['train'].column_names)
eval_ds = ds['test'].map(fmt, remove_columns=ds['test'].column_names)
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

## 7. Train (REDUCED MEMORY)

In [ ]:
# Reduced settings for T4 GPU
args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    bf16=True,
    max_grad_norm=0.3,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024
)

print("🚀 Training (slower but fits in memory)...")
trainer.train()
print("✅ Done!")

## 8. Save

In [ ]:
trainer.model.save_pretrained("./llama-3.2-3b-lora")
tokenizer.save_pretrained("./llama-3.2-3b-lora")
print("✅ Download llama-3.2-3b-lora folder!")